In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[],
    store=store,
)

In [3]:
user_id = "user_001"
application_context = "personal_assistant"

namespace = (user_id, application_context)

In [4]:
store.put(
    namespace,
    "memory_001",
    {
        "facts": [
            "사용자는 아아를 선호함",
            "사용자는 랭체인을 공부하고 있음",
            "사용자는 AI 애플리케이션 사이드 프로젝트를 진행 중"
        ]
    },
)


In [5]:
item = store.get(namespace, "memory_001")
print("저장된 장기 기억:", item)

저장된 장기 기억: Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 랭체인을 공부하고 있음', '사용자는 AI 애플리케이션 사이드 프로젝트를 진행 중']}, created_at='2026-06-06T05:31:09.727196+00:00', updated_at='2026-06-06T05:31:09.727196+00:00')


In [6]:
store.put(
    namespace,
    "memory_002",
    {
        "facts": [
            "사용자는 매일 아침 7시에 일어남",
            "진행중인 사이드 프로젝트는 학습용 챗봇 프로그램임"
        ]
    },
)

In [7]:
item = store.get(namespace, "memory_001")
item

Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 랭체인을 공부하고 있음', '사용자는 AI 애플리케이션 사이드 프로젝트를 진행 중']}, created_at='2026-06-06T05:31:09.727196+00:00', updated_at='2026-06-06T05:31:09.727196+00:00')

In [8]:
item = store.get(namespace, "memory_002")
item

Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 매일 아침 7시에 일어남', '진행중인 사이드 프로젝트는 학습용 챗봇 프로그램임']}, created_at='2026-06-06T05:31:09.773192+00:00', updated_at='2026-06-06T05:31:09.773192+00:00')

In [9]:
store.search(namespace)

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아를 선호함', '사용자는 랭체인을 공부하고 있음', '사용자는 AI 애플리케이션 사이드 프로젝트를 진행 중']}, created_at='2026-06-06T05:31:09.727196+00:00', updated_at='2026-06-06T05:31:09.727196+00:00', score=None),
 Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 매일 아침 7시에 일어남', '진행중인 사이드 프로젝트는 학습용 챗봇 프로그램임']}, created_at='2026-06-06T05:31:09.773192+00:00', updated_at='2026-06-06T05:31:09.773192+00:00', score=None)]

In [10]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

In [11]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_memory(request, handler):
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name
    memories = request.runtime.store.search((current_user, current_app))

    memory_content = "기록된 정보 없음"
    if memories:
        # 검색된 메모리들을 텍스트 변환
        extracted_facts = []
        for item in memories:
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
        memory_content = "\n- ".join(extracted_facts)

    system_message=f"사용자 관련 장기 메모리 : {memory_content}"
    request = request.override(system_prompt=system_message)
    return handler(request)

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    store=store, # store 연결
    context_schema=Context,
    middleware=[inject_memory] # 미들웨어 장착
)

In [13]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

In [14]:
response

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='d80706fe-6752-4515-939a-aa197aec43a8'),
  AIMessage(content=[{'type': 'text', 'text': "사용자님에 대해 제가 기억하고 있는 정보는 다음과 같습니다!\n\n*   **음료 취향:** 시원한 '아아(아이스 아메리카노)'를 선호하십니다.\n*   **공부 중인 분야:** '랭체인(LangChain)'을 공부하고 계십니다.\n*   **사이드 프로젝트:** 학습용 챗봇 프로그램을 개발하는 AI 애플리케이션 사이드 프로젝트를 진행 중이십니다.\n*   **생활 습관:** 매일 아침 7시에 기상하십니다.\n\n혹시 이 내용 중에서 업데이트하거나 추가하고 싶은 정보가 있으신가요? 프로젝트나 공부하시면서 도움이 필요하시면 언제든 말씀해 주세요! 아아 한 잔 마시면서 힘내시길 바랍니다. :)", 'extras': {'signature': 'EjQKMgEMOdbHp1V7isyFOPcBmuHeuWjKYsKlAT2l0ZN0uWVbEXJoo1lfozN/CtXPwGTwD6EJ'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9b69-ea42-76f0-aa2e-9333ccc4c7f8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 167, 'total_tokens': 250, 'input_token_details': {'cache_rea